In [ ]:
algorithm_list = ["mine", "mine", "mine"]
Nit_list = [700, 50, 50]  # iterations for algorithm_list
algorithm_list_pc = ["mine", "ER", "ER"]
Nit_list_pc = [700, 50, 50]


offset1 = 45
offset2 = 55


for ii,magnet_A in enumerate(scan_axis[offset1:offset2] ):
    print('test')
    ii = offset1 + ii
    for photonlevel in [-1000]:
        
        if ii >= 0:
            print(ii,magnet_A)
        
            image=load_images(im_id, file_indices = [ii])
            
            image=np.squeeze(image)-dark
            image[np.where(image<photonlevel)]=0
            image=np.average(image, axis=0)
            
    
    
            for crop in [400][::-1]:
                roicrop=np.s_[crop:-crop, crop:-crop]
                mask_pixel2=mask_pixel[roicrop]
                
                worker_dict = worker(image, topo, crop=crop)
        
                pos=worker_dict["im_c"].copy()
                neg=worker_dict["topo_c"].copy()
    
                
                # Executes the algorithm
                for offset_vmin in [-1]:
                    for offset in [0]:
                        for dilation in [0]:
                            supportmask2=mask_lib.scale_mask(supportmask,mask_pixel2.shape)
                            for Startimage in [None ]:
                
                            
                                supportmask=1.*(plt.imread("supportmask_00_2.png")[:,:,0]==1)
                                from skimage.draw import disk
                                yy,xx=disk(np.array(supportmask.shape)//2,4)
                                supportmask[yy,xx]=1
                                if dilation>0:
                                    supportmask=1.*scipy.ndimage.binary_dilation(supportmask, iterations=dilation)
                                if dilation<0:
                                    supportmask=1.*scipy.ndimage.binary_erosion(supportmask, iterations=dilation)                    
                    
                                supportmask2=mask_lib.scale_mask(supportmask,mask_pixel2.shape)
                                
                                #recon = fth.reconstruct(
                                #    fth.propagate(holo, prop_dist * 1e-6, experimental_setup=experimental_setup)
                                #    * np.exp(1j * phase)
                                #)
                                #Startimage = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(supportmask * recon)))
                                #Startimage = None
                                Startgamma = None
                                
                                (
                                    retrieved_p,
                                    retrieved_n,
                                    retrieved_p_pc,
                                    retrieved_n_pc,
                                    bsmask_p,
                                    bsmask_n,
                                    gamma_p,
                                    gamma_n,
                                ) = phase_retrieval(
                                    pos+offset,
                                    neg+offset,
                                    mask_pixel2,
                                    supportmask2,
                                    vmin=offset_vmin,
                                    Startimage=Startimage,
                                    Startgamma=Startgamma,
                                    algorithm_list=algorithm_list,
                                    Nit_list=Nit_list,
                                    algorithm_list_pc=algorithm_list_pc ,
                                    Nit_list_pc=Nit_list_pc
                                )
                    
                                prop_dist_cdi=2.05
                                phase_cdi=-0.76
            
            
                                roi_s=np.s_[supportmask2.shape[0]//5:2*supportmask2.shape[0]//5,supportmask2.shape[0]//5:2*supportmask2.shape[0]//5]
                                
                                # Reconstruct images with new parameter
                                p_pc = fth.reconstructCDI(
                                    fth.propagate(
                                        retrieved_p_pc,
                                        prop_dist_cdi * 1e-6,
                                        experimental_setup=experimental_setup,
                                    )
                                ) * np.exp(1j * phase_cdi)
                                
                                n_pc = fth.reconstructCDI(
                                    fth.propagate(
                                        retrieved_n_pc,
                                        prop_dist_cdi * 1e-6,
                                        experimental_setup=experimental_setup,
                                    )
                                ) * np.exp(1j * phase_cdi)
            
                                recon = p_pc - n_pc
                                
            
            
                                title = get_title(None, im_id, topo_id, CDI=True)
                                plot_recon(
                                    recon[roi_s]*supportmask2[roi_s] , title,# scale_mask=shrink_mask[roi_cdi_s]
                                )
                                
                                # Save images
                                fname = join(
                                    folder_general,
                                    "Recon_ImId_%s_RefId_%s_cdi_field=%0.4f_diff_%s_crop=%d_photonlevel=%0.1f_%d_%d.png" % (im_id, topo_id,  magnet_A,USER,crop,photonlevel,offset, offset_vmin),
                                )
                                print("Saving: %s" % fname)
                                plt.savefig(fname, bbox_inches="tight", transparent=False)

                                save_cdi_h5(fname=fname, magnet_A=magnet_A)




